# ML-07 — Baseline Segmentation & Rule Heuristic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AzizullahMemonAi/FlyRank-ML-Assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Transparent 3-tier rules-based heuristic baseline.

## 1. Baseline Heuristic Logic

Before training machine learning clustering models, we establish a transparent 3-tier rule baseline:
- **Tier 0 (High Visibility / Top Rank):** `impressions_90d >= 500` AND `avg_position` in `[1, 15]`
- **Tier 1 (Mid Potential):** `impressions_90d >= 50`
- **Tier 2 (Low / Zombie):** All remaining inventory

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

def assign_baseline(row):
    if row['impressions_90d'] >= 500 and row['avg_position'] > 0 and row['avg_position'] <= 15:
        return 0
    elif row['impressions_90d'] >= 50:
        return 1
    else:
        return 2

df['baseline_tier'] = df.apply(assign_baseline, axis=1)
print('Baseline Tier Distribution:')
print(df['baseline_tier'].value_counts(normalize=True).round(3))


Baseline Tier Distribution:
baseline_tier
1    0.432
0    0.341
2    0.227
Name: proportion, dtype: float64


## 2. Baseline Coherence Evaluation

We measure the silhouette score of the rule baseline across standardized feature space.

In [1]:
# Compute baseline silhouette score on standardized feature space
features = pd.DataFrame({
    'log_imp': np.log1p(df['impressions_90d']),
    'log_clicks': np.log1p(df['clicks_90d']),
    'pos': df['avg_position'].replace(0, 100),
    'ctr': df['ctr'].fillna(0),
    'eng': df['engagement_rate'].fillna(0),
    'scroll': df['scroll_rate'].fillna(0),
    'age': np.log1p(df['content_age_days'])
})
X_scaled = StandardScaler().fit_transform(features)
np.random.seed(42)
sub_idx = np.random.choice(len(X_scaled), size=5000, replace=False)
base_sil = silhouette_score(X_scaled[sub_idx], df['baseline_tier'].iloc[sub_idx])
print(f'Rule Baseline Silhouette Score: {base_sil:.4f}')


Rule Baseline Silhouette Score: 0.0638


## 3. Baseline Limitations

The rule baseline achieves a poor silhouette score of **0.0638**, indicating severe overlap between tiers. It ignores engagement depth, scroll rates, content aging decay, and keyword context, failing to distinguish between high-potential striking distance pages and decaying legacy pages.